<a href="https://colab.research.google.com/github/gano0802/BirdCLEF2026/blob/main/BC2026_SelfKD_SED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!unzip -q /content/drive/MyDrive/kaggle/Birdclef2026/input/birdclef-2026-waveform-cache.zip -d /content

error:  zipfile read error


# S1 -- IMPORTS + CONFIG

In [2]:
EXP = "exp13"

In [3]:
# =================================================================
# S1 -- IMPORTS + CONFIG
# =================================================================
import os, sys, time, json, pickle, gc, random, math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
import torchaudio
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.special import expit as sigmoid_np
from tqdm.notebook import tqdm, trange
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True # False
torch.backends.cudnn.benchmark = False # True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name()}")

# ------------------------------------------------------------------
# Notebook Mode
# -----------------------------------------------------------------
MODE = "train"  # "train" or "infer"

if MODE == 'train':    #Kaggle struggles with full training pipeline
    DEBUG = False
else:
    DEBUG = False

# ------------------------------------------------------------------
# Paths -- Kaggle dataset layout
# -----------------------------------------------------------------
COMP_DIR = Path("/content/drive/MyDrive/kaggle/Birdclef2026")
WAVEFORM_CACHE_DIR = Path("/content/waveform_cache")
PERCH_ONNX_PATH = Path("/content/drive/MyDrive/kaggle/Birdclef2026/input/perch-v2-no-dft-onnx/perch_v2_no_dft.onnx")

LABELS_PATH     = COMP_DIR / "input" / "train_soundscapes_labels.csv"
TAXONOMY_PATH   = COMP_DIR / "input" / "taxonomy.csv"
SAMPLE_SUB_PATH = COMP_DIR / "input" / "sample_submission.csv"
TEST_DIR        = COMP_DIR / "input" / "test_soundscapes"

OUT_DIR = Path("/content/drive/MyDrive/kaggle/Birdclef2026/outputs")

NUM_CLASSES = 234
SR = 32000

# --- Duration ---
TRAIN_DURATION = 5    # seconds
VAL_DURATION   = 5    # always 5s for competition eval
TRAIN_SAMPLES  = SR * TRAIN_DURATION
VAL_SAMPLES    = SR * VAL_DURATION

N_FOLDS = 5

# --- Mel spectrogram ---
N_FFT      = 2048
HOP_LENGTH = 512
N_MELS     = 256
FMIN       = 20
FMAX       = 16000

# --- Model ---
BACKBONE_NAME = "tf_efficientnet_b0.ns_jft_in1k"
# "tf_efficientnet_b0.ns_jft_in1k"
# "seresnext26t_32x4d", "regnety_008.pycls_in1k"

# --- Perch distillation ---
USE_PERCH_DISTILL = True
PERCH_EMBED_DIM   = 1536
ALPHA_DISTILL     = 1.0   # MSE loss weight

# --- Training ---
FOLDS  = [0] # [0, 1, 2, 3, 4]
EPOCHS = 25
BATCH  = 16 if DEBUG else 64        # used 64 locally, 16 for Kaggle
LR     = 5e-4
MIN_LR = 1e-5
WD     = 1e-4
WARMUP_EPOCHS = 2

# --- Upsampling ---
MIN_SAMPLE = 20

# --- Augmentation ---
AUG_PROB = 0.5
AUG_GAIN_DB_RANGE      = (-6.0, 6.0)
AUG_NOISE_SNR_DB_RANGE = (10.0, 30.0)
AUG_SHIFT_SAMPLES_MAX  = int(SR * 0.5)

# --- MixUp ---
USE_FOCAL_MIXUP    = True
MIXUP_PROB         = 0.5
MIXUP_ALPHA        = 0.4
MIXUP_HARD         = True    # union labels (hard) vs weighted blend (soft)

USE_FOCAL_SC_MIXUP     = True
FOCAL_SC_MIXUP_PROB    = 0.5
FOCAL_SC_MIXUP_ALPHA   = 0.4

# --- FreqMixStyle (disabled by default) ---
FREQ_MIXSTYLE_PROB  = 0.0
FREQ_MIXSTYLE_ALPHA = 0.1

# --- SpecAugment ---
FREQ_MASK_PARAM = 10
TIME_MASK_PARAM = 10
NUM_FREQ_MASKS  = 1
NUM_TIME_MASKS  = 2

# --- Source weights ---
USE_FOCAL           = True
USE_FOCAL_SECONDARY = True
USE_LABELED_SC      = True

ACTIVE_SOURCES = ["focal", "sc"]
SHARES = {"focal": 0.9, "sc": 0.1}
SOURCE_WEIGHTS = {
    "focal":         1.0,
    "focal_missing": 0.0,
    "sc":            1.0,
}

# --- Pseudo label ---
USE_PSEUDO_LABEL = True

print(f"Backbone: {BACKBONE_NAME}")
print(f"Train duration: {TRAIN_DURATION}s | Mel: {N_MELS} mels, n_fft={N_FFT}, hop={HOP_LENGTH}")
print(f"Distillation: {'ON' if USE_PERCH_DISTILL else 'OFF'} (alpha={ALPHA_DISTILL})")
print(f"Self Distillation: {'ON' if USE_PSEUDO_LABEL else 'OFF'}")
print(f"Batch: {BATCH} | Epochs: {EPOCHS} | Folds: {FOLDS}")

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Backbone: tf_efficientnet_b0.ns_jft_in1k
Train duration: 5s | Mel: 256 mels, n_fft=2048, hop=512
Distillation: ON (alpha=1.0)
Self Distillation: ON
Batch: 64 | Epochs: 25 | Folds: [0]


In [4]:
os.makedirs(OUT_DIR / f"{EXP}", exist_ok=True)

In [5]:
if MODE == "train":
    !uv pip install -q timm torchaudio onnxscript onnx onnxruntime-gpu

# S2 -- LOAD DATA

In [6]:
# =================================================================
# S2 -- LOAD DATA
# =================================================================

# --- Label ordering from sample_submission (defines column order) ---
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL2IDX = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
taxonomy = pd.read_csv(TAXONOMY_PATH)
label_to_taxon = dict(zip(taxonomy["primary_label"].astype(str),
                          taxonomy["class_name"].astype(str)))
TAXON_MASKS = {t: np.array([i for i, l in enumerate(PRIMARY_LABELS)
                            if label_to_taxon.get(l, "") == t])
               for t in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]}

# --- Focal recording metadata ---
audio_cache_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "audio_cache_meta.csv")
train_df = pd.read_csv(COMP_DIR / "input" / "train.csv")
audio_cache_meta = audio_cache_meta.merge(
    train_df[["filename", "secondary_labels"]], on="filename", how="left"
)
audio_cache_meta = audio_cache_meta[
    audio_cache_meta["primary_label"].isin(LABEL2IDX)
].reset_index(drop=True)
print(f"Focal audio cache: {len(audio_cache_meta)} entries")

# --- Soundscape window metadata ---
sc_cache_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "soundscape_cache_meta.csv")
sc_cache_meta["label_list"] = sc_cache_meta["label_list"].apply(
    lambda x: x.split(";") if isinstance(x, str) else []
)
print(f"Soundscape cache: {len(sc_cache_meta)} windows")

# --- Build soundscape label matrix from ground truth ---
sc_labels_raw = pd.read_csv(LABELS_PATH).drop_duplicates()
sc_labels_raw["start_sec"] = pd.to_timedelta(sc_labels_raw["start"]).dt.total_seconds().astype(int)

Y_SC = np.zeros((len(sc_cache_meta), NUM_CLASSES), dtype=np.float32)
for i, row in sc_cache_meta.iterrows():
    matches = sc_labels_raw[
        (sc_labels_raw["filename"] == row["filename"]) &
        (sc_labels_raw["start_sec"] == row["start_sec"])
    ]
    for _, m in matches.iterrows():
        for lbl in str(m["primary_label"]).split(";"):
            lbl = lbl.strip()
            if lbl in LABEL2IDX:
                Y_SC[i, LABEL2IDX[lbl]] = 1.0

labeled_sc_mask = Y_SC.sum(axis=1) > 0
print(f"Soundscape labels: {labeled_sc_mask.sum()}/{len(Y_SC)} windows labeled, "
      f"{int(Y_SC.sum())} positives, {int((Y_SC.sum(axis=0) > 0).sum())} species")

# =================================================================
# FOLD ASSIGNMENT
# =================================================================

# --- Focal: StratifiedKFold by species ---
audio_for_split = audio_cache_meta.drop_duplicates("original_idx").reset_index(drop=True)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
audio_for_split["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(audio_for_split, audio_for_split["primary_label"])):
    audio_for_split.loc[val_idx, "fold"] = fold
audio_cache_meta = audio_cache_meta.merge(
    audio_for_split[["original_idx", "fold"]], on="original_idx", how="left"
)
print(f"\nFocal fold distribution:\n{audio_cache_meta['fold'].value_counts().sort_index()}")

# --- Soundscape: file-level folds, all 66 files distributed ---
# All files including S22 participate in CV for maximum species coverage
# (46 multi-fold species vs 32 with S22 holdout). The non_s22_mask_sc
# filter in evaluation still excludes S22 windows from the primary metric.
from sklearn.model_selection import GroupKFold

sc_files = sc_cache_meta[["filename", "site"]].drop_duplicates().reset_index(drop=True)
gkf = GroupKFold(n_splits=N_FOLDS)
sc_files["fold"] = -1
for fold, (_, val_idx) in enumerate(gkf.split(sc_files, groups=sc_files["filename"])):
    sc_files.loc[sc_files.index[val_idx], "fold"] = fold

file_to_fold = dict(zip(sc_files["filename"], sc_files["fold"]))
sc_cache_meta["fold"] = sc_cache_meta["filename"].map(file_to_fold).fillna(-1).astype(int)
print(f"\nSoundscape fold distribution:")
print(sc_cache_meta["fold"].value_counts().sort_index())
# =================================================================
# UPSAMPLE RARE SPECIES
# =================================================================
counts = audio_cache_meta["primary_label"].value_counts()
rare_species = counts[counts < MIN_SAMPLE].index
extra_rows = []
for sp in rare_species:
    sp_rows = audio_cache_meta[audio_cache_meta["primary_label"] == sp]
    n_copies = int(np.ceil(MIN_SAMPLE / len(sp_rows))) - 1
    for _ in range(n_copies):
        extra_rows.append(sp_rows)

n_before = len(audio_cache_meta)
if extra_rows:
    audio_cache_meta = pd.concat([audio_cache_meta] + extra_rows, ignore_index=True)
print(f"\nUpsampled {len(rare_species)} rare species (min={MIN_SAMPLE}): "
      f"{n_before} -> {len(audio_cache_meta)} samples")

# Non-S22 mask for evaluation (S22 is a site with known label noise)
sc_sites = sc_cache_meta["site"].values
non_s22_mask_sc = sc_sites != "S22"
print(f"S22: {(~non_s22_mask_sc).sum()}, non-S22: {non_s22_mask_sc.sum()}")
print("OK Data loaded")

Focal audio cache: 35549 entries
Soundscape cache: 739 windows
Soundscape labels: 739/739 windows labeled, 3122 positives, 75 species

Focal fold distribution:
fold
0    7110
1    7110
2    7110
3    7110
4    7109
Name: count, dtype: int64

Soundscape fold distribution:
fold
0    155
1    137
2    146
3    149
4    152
Name: count, dtype: int64

Upsampled 36 rare species (min=20): 35549 -> 36135 samples
S22: 477, non-S22: 262
OK Data loaded


In [7]:
if DEBUG:
    EPOCHS = 7
    FOLDS = [0]
    audio_cache_meta = audio_cache_meta.groupby("primary_label").head(3).reset_index(drop=True)
    sc_cache_meta = sc_cache_meta.head(50)
    Y_SC = Y_SC[:50]
    non_s22_mask_sc = non_s22_mask_sc[:50]
    print(f"DEBUG MODE: {len(audio_cache_meta)} focal, {len(sc_cache_meta)} sc, "
          f"{EPOCHS} epoch, folds={FOLDS}")

# S3 — Model Architecture

* SED (Sound Event Detection) head design  
Instead of Global Average Pooling (GAP), which dilutes a brief vocalization across the full 5-second window, the SED head makes per-frame predictions and aggregates them via learned attention weights. A species that calls for 0.3 seconds gets a sharp attention spike at those frames, rather than being averaged with 4.7 seconds of background.

* Distillation head  
A separate branch applies GAP to the raw backbone features and projects them to Perch’s 1536-d embedding space. This branch receives all backbone gradients via MSE loss against frozen Perch embeddings. The SED classification head operates on h.detach() — it only shapes the attention and frame-level classifier, never pulling the backbone away from Perch’s representation.

* Why stop-gradient?  
Without the stop-gradient, the classification loss and distillation loss would fight over the backbone’s feature representation. With it, the backbone is purely a “Perch student” — learning to reproduce Perch’s rich 14,795-species embedding geometry. The SED head then learns to classify using those frozen-quality features.

* Two model variants are implemented:  

    * V1: Simple frequency-mean pooling + attention block
    * V2 (default): Generalized Mean (GeM) frequency pooling + 512-d bottleneck + attention block — inspired by the BirdCLEF 2021 first-place solution

In [8]:
# =================================================================
# S3 -- EVAL UTILITIES + MEL TRANSFORM + SED MODELS
# =================================================================

def compute_macro_auc(y_true, y_pred, mask=None, class_mask=None):
    """Macro-averaged AUC across evaluable species."""
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    if class_mask is not None:
        y_true, y_pred = y_true[:, class_mask], y_pred[:, class_mask]
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        try:
            aucs.append(roc_auc_score(col, y_pred[:, c]))
        except ValueError:
            continue
    return (np.mean(aucs) if aucs else float("nan")), len(aucs)

def full_eval(y_true, y_pred, ns22, tm):
    r = {}
    a, n = compute_macro_auc(y_true, y_pred)
    r["macro_auc_all"], r["n_all"] = round(a, 4), n
    a, n = compute_macro_auc(y_true, y_pred, mask=ns22)
    r["non_s22_macro"], r["n_ns22"] = round(a, 4), n
    for t, cm in tm.items():
        a, n = compute_macro_auc(y_true, y_pred, mask=ns22, class_mask=cm)
        r[f"non_s22_{t}"] = round(a, 4)
    return r

# ------------------------------------------------------------------
# GPU Mel Spectrogram
# ------------------------------------------------------------------
class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0,
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)

    def forward(self, waveform):
        return self.db_transform(self.mel_spec(waveform))

# ------------------------------------------------------------------
# GPU SpecAugment
# # ------------------------------------------------------------------
class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=FREQ_MASK_PARAM)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=TIME_MASK_PARAM)

    def forward(self, mel):
        for _ in range(NUM_FREQ_MASKS):
            mel = self.freq_mask(mel)
        for _ in range(NUM_TIME_MASKS):
            mel = self.time_mask(mel)
        return mel

# ------------------------------------------------------------------
# Frozen Perch teacher -- ONNX inference, no gradients
# ------------------------------------------------------------------
import onnxruntime as ort

class PerchTeacher:
    """Frozen Perch v2 via ONNX. Takes 5s waveforms, returns 1536-d embeddings.
    The teacher is never updated -- it provides a stable distillation target."""

    def __init__(self, onnx_path, device_str="cuda"):
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] \
            if device_str == "cuda" else ["CPUExecutionProvider"]
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        self._out_names = [o.name for o in self.session.get_outputs()]
        self._embed_idx = None
        for i, o in enumerate(self.session.get_outputs()):
            if o.shape and o.shape[-1] == PERCH_EMBED_DIM:
                self._embed_idx = i
                break
        if self._embed_idx is None:
            self._embed_idx = 1
        print(f"Perch ONNX loaded: embed_idx={self._embed_idx}")

    @torch.no_grad()
    def embed(self, waveforms_5s):
        """waveforms_5s: (B, 160000) float32, returns (B, 1536) embeddings."""
        wav_np = waveforms_5s.cpu().numpy()
        results = self.session.run(None, {self.input_name: wav_np})
        return torch.from_numpy(results[self._embed_idx]).float()

# ------------------------------------------------------------------
# Distillation head: GAP + Linear to Perch embedding space
# ------------------------------------------------------------------
class DistillHead(nn.Module):
    """Projects backbone features to Perch's 1536-d space via GAP + Linear."""
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)

    def forward(self, feature_map):
        gap = feature_map.mean(dim=[2, 3])   # (B, C, F, T) -> (B, C)
        return self.proj(gap)                 # (B, embed_dim)

# ------------------------------------------------------------------
# SED Model V2: GeMFreq + bottleneck + AttBlock (recommended)
# ------------------------------------------------------------------
class GeMFreqPool(nn.Module):
    """Generalized Mean pooling over frequency. Learnable p starts at 3.0
    (sharper than mean, softer than max). Lets the model emphasize
    frequency bands where species vocalize."""
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)

class BirdSEDModel(nn.Module):
    """SED model with 1st-place-inspired design: https://www.kaggle.com/code/nikitababich/birdclef2025-1st-place-inference
    - GeMFreq pooling (learnable, sharper than mean)
    - 512-dim bottleneck with dropout
    - Attention-weighted clip logits from frame logits
    - Distillation: GAP+Linear branch for MSE to Perch
    - Stop gradient: backbone trains from distillation only
    """
    def __init__(self, backbone_name=BACKBONE_NAME, num_classes=NUM_CLASSES,
                 drop_path_rate=0.1, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, in_chans=1,
            num_classes=0, global_pool="", drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            n_tf = TRAIN_SAMPLES // HOP_LENGTH + 1
            dummy = torch.randn(1, 1, N_MELS, n_tf)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]
            print(f"V2 backbone: {tuple(feat.shape)}  (C={self.backbone_dim})")

        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        nn.init.xavier_uniform_(self.att.weight)
        nn.init.xavier_uniform_(self.cla.weight)
        self.att.bias.data.fill_(0.)
        self.cla.bias.data.fill_(0.)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        distill_emb = None
        if return_distill and hasattr(self, 'distill_head'):
            # h_for_distill = F.dropout(h, p=0.3, training=self.training) # add
            distill_emb = self.distill_head(h)
            # distill_emb = self.distill_head(h_for_distill)

        # Stop gradient: SED head doesn't update the backbone
        h_cls = h.detach() if USE_PERCH_DISTILL else h

        h_cls = self.gem_freq(h_cls)            # (B, C, T)
        h_cls = h_cls.permute(0, 2, 1)          # (B, T, C)
        h_cls = self.dense(h_cls)               # (B, T, 512)
        h_cls = h_cls.permute(0, 2, 1)          # (B, 512, T)

        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise_logits = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise_logits, dim=2)

        fw = framewise_logits.permute(0, 2, 1) if return_framewise else None
        if return_framewise and return_distill: return clip_logits, fw, distill_emb
        elif return_framewise: return clip_logits, fw
        elif return_distill: return clip_logits, distill_emb
        return clip_logits

def make_model():
        return BirdSEDModel(BACKBONE_NAME).to(device)

print("OK Model definitions ready")

OK Model definitions ready


# S4 — Data Pipeline

* Waveform loading¶  
The waveform cache stores audio as int16 PyTorch tensors (half the size of float32). Each focal recording and soundscape window is pre-sliced and saved as a separate .pt file, indexed by the metadata CSVs.

* Augmentation pipeline  
Three simple waveform augmentations applied with probability 0.5 each:

    1. Gain jitter (±6 dB) — simulates varying recording distances
    2. Additive noise (10–30 dB SNR) — simulates environmental noise
    3. Time shift (±0.5s) — simulates temporal misalignment
    4. MixUp strategy
* Two MixUp variants run independently:  

    * Focal-Focal MixUp: Blends two focal recordings with Beta(0.4, 0.4) weighting. Labels are unioned (hard MixUp).
    * Focal-Soundscape MixUp: Blends a focal recording with a labeled soundscape window, bridging the domain gap between clean focal audio and noisy real-world soundscapes.

In [9]:
# =================================================================
# S4 -- DATA PIPELINE
# =================================================================

def load_int16(path):
    """Load int16 waveform tensor to float32 in [-1, 1]."""
    waveform_int16 = torch.load(path, map_location="cpu")
    return waveform_int16.float() / 32767.0

_FC = {}
def load_focal(p):
    """Load focal waveform with simple LRU cache."""
    if p in _FC: return _FC[p]
    pp = WAVEFORM_CACHE_DIR / p
    if not pp.exists(): return None
    a = load_int16(pp).numpy()
    if len(_FC) >= 2000:
        _FC.pop(next(iter(_FC)))
    _FC[p] = a
    return a

_SC_CACHE = {}
def load_sc_waveform_from(cache_dir, cache_file):
    """Load a soundscape waveform with LRU cache."""
    key = str(cache_dir / cache_file)
    if key in _SC_CACHE: return _SC_CACHE[key]
    pp = cache_dir / cache_file
    if not pp.exists(): return None
    a = load_int16(pp).numpy()
    if len(_SC_CACHE) >= 200:
        _SC_CACHE.pop(next(iter(_SC_CACHE)))
    _SC_CACHE[key] = a
    return a

def extract_chunk_np(waveform, start_sample, n_samples):
    """Extract a chunk, left-padding if the recording is too short."""
    total = len(waveform)
    if total <= n_samples:
        return np.pad(waveform, (n_samples - total, 0))
    end = start_sample + n_samples
    if end > total:
        start_sample = max(0, total - n_samples)
    return waveform[start_sample:start_sample + n_samples]

def apply_aug(w):
    """Simple waveform augmentation: gain jitter + noise + shift."""
    if np.random.random() < AUG_PROB:
        w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
    if np.random.random() < AUG_PROB:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))
    return w

# ------------------------------------------------------------------
# Build soundscape MixUp pool (labeled windows only)
# ------------------------------------------------------------------
sc_mixup_sources = []
_sc_file_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "soundscape_file_meta.csv")
_sc_file_dict = dict(zip(_sc_file_meta["filename"], _sc_file_meta["cache_file"]))
_labeled_rows = []
for i in range(len(sc_cache_meta)):
    row = sc_cache_meta.iloc[i]
    if Y_SC[i].sum() > 0:
        cf = _sc_file_dict.get(row["filename"])
        if cf is not None:
            _labeled_rows.append({
                "filename": row["filename"], "start_sec": int(row["start_sec"]),
                "cache_file": cf, "label_idx": i, "fold": int(row.get("fold", -1)),
            })
if _labeled_rows:
    _labeled_meta = pd.DataFrame(_labeled_rows)
    sc_mixup_sources.append((WAVEFORM_CACHE_DIR, _labeled_meta, Y_SC))
    print(f"SC MixUp pool: {len(_labeled_meta)} labeled windows")

# ------------------------------------------------------------------
# FocalDS -- with Focal-Focal AND Focal-Soundscape MixUp
# ------------------------------------------------------------------
class FocalDS(Dataset):
    """Focal recording dataset. Returns (waveform, label, weight, mask, source_tag)."""
    def __init__(self, df, l2i, secondary_lookup=None,
                 sc_mixup_sources=None, fold_k=None, aug=False):
        self.df, self.l2i, self.aug = df.reset_index(drop=True), l2i, aug
        self.secondary_lookup = secondary_lookup
        self.sc_mixup_sources = sc_mixup_sources
        self.fold_k = fold_k

    def __len__(self): return len(self.df)

    def _load_chunk(self, r):
        w = load_focal(r["cache_file"])
        if w is None: return None, None
        if self.aug:
            start = np.random.randint(0, max(1, len(w) - TRAIN_SAMPLES + 1)) if len(w) > TRAIN_SAMPLES else 0
        else:
            start = int(r.get("start_sec", 0)) * SR
        ch = extract_chunk_np(w, start, TRAIN_SAMPLES)
        lb = np.zeros(NUM_CLASSES, dtype=np.float32)
        if str(r["primary_label"]) in self.l2i:
            lb[self.l2i[str(r["primary_label"])]] = 1.0
        if self.secondary_lookup is not None and "original_idx" in self.df.columns:
            for s in self.secondary_lookup.get(int(r["original_idx"]), []):
                if s in self.l2i: lb[self.l2i[s]] = 1.0
        return ch, lb

    def __getitem__(self, i):
        r1 = self.df.iloc[i]
        ch1, lb1 = self._load_chunk(r1)
        if ch1 is None:
            return (torch.zeros(1, TRAIN_SAMPLES), torch.zeros(NUM_CLASSES),
                    torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal_missing", torch.tensor(0.0, dtype=torch.float32))

        # Focal-Focal MixUp
        if USE_FOCAL_MIXUP and self.aug and np.random.random() < MIXUP_PROB:
            ch2 = None
            for _ in range(3):
                j = np.random.randint(len(self.df))
                ch2, lb2 = self._load_chunk(self.df.iloc[j])
                if ch2 is not None: break
            if ch2 is not None:
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                ch_mix = (lam * ch1 + (1 - lam) * ch2).astype(np.float32)
                if self.aug: ch_mix = apply_aug(ch_mix)
                lb = np.maximum(lb1, lb2) if MIXUP_HARD else (lam * lb1 + (1 - lam) * lb2)
                return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                        torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal", torch.tensor(0.0, dtype=torch.float32))

        # Focal-Soundscape MixUp
        if (USE_FOCAL_SC_MIXUP and self.aug and self.sc_mixup_sources
                and np.random.random() < FOCAL_SC_MIXUP_PROB):
            src_idx = np.random.randint(len(self.sc_mixup_sources))
            cache_dir, meta_df_sc, labels = self.sc_mixup_sources[src_idx]
            eligible = meta_df_sc[meta_df_sc["fold"] != self.fold_k] if self.fold_k is not None else meta_df_sc
            if len(eligible) > 0:
                sc_row = eligible.iloc[np.random.randint(len(eligible))]
                sc_wav = load_sc_waveform_from(cache_dir, sc_row["cache_file"])
                if sc_wav is not None and len(sc_wav) >= TRAIN_SAMPLES:
                    sc_chunk = extract_chunk_np(sc_wav, int(sc_row["start_sec"]) * SR, TRAIN_SAMPLES)
                    lam = np.random.beta(FOCAL_SC_MIXUP_ALPHA, FOCAL_SC_MIXUP_ALPHA)
                    ch_mix = (lam * ch1 + (1 - lam) * sc_chunk).astype(np.float32)
                    if self.aug: ch_mix = apply_aug(ch_mix)
                    lb_sc = labels[int(sc_row["label_idx"])].astype(np.float32)
                    lb = np.maximum(lb1, lb_sc) if MIXUP_HARD else lam * lb1 + (1 - lam) * lb_sc
                    return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                            torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal", torch.tensor(0.0, dtype=torch.float32))

        # No MixUp
        if self.aug: ch1 = apply_aug(ch1)
        return (torch.from_numpy(ch1.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(lb1),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal", torch.tensor(0.0, dtype=torch.float32))

# ------------------------------------------------------------------
# ScDS -- Labeled soundscape windows
# ------------------------------------------------------------------
class ScDS(Dataset):
    def __init__(self, Y, sc_df, aug=False):
        self.Y, self.df, self.aug = Y, sc_df.reset_index(drop=True), aug
    def __len__(self): return len(self.Y)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        wav_full = load_sc_waveform_from(WAVEFORM_CACHE_DIR, row.get("cache_file")) \
                   if row.get("cache_file") else None
        if wav_full is None:
            wav_t = torch.zeros(1, TRAIN_SAMPLES)
        else:
            chunk = extract_chunk_np(wav_full, int(row["start_sec"]) * SR, TRAIN_SAMPLES)
            if self.aug: chunk = apply_aug(chunk)
            wav_t = torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0)
        return (wav_t, torch.from_numpy(self.Y[i].astype(np.float32)),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "sc", torch.tensor(1.0, dtype=torch.float32))

# ------------------------------------------------------------------
# Load focal secondary labels
# ------------------------------------------------------------------
focal_secondary_labels = None
if USE_FOCAL_SECONDARY:
    focal_secondary_labels = {}
    for idx, row in train_df.iterrows():
        sec = row.get("secondary_labels", "")
        if pd.isna(sec) or sec in ("", "[]"): continue
        try:
            sec_list = eval(sec) if isinstance(sec, str) else []
        except: continue
        valid = [s for s in sec_list if s in LABEL2IDX]
        if valid: focal_secondary_labels[idx] = valid
    print(f"Focal secondary labels: {len(focal_secondary_labels)} files")

# ------------------------------------------------------------------
# Multi-source batch sampler
# ------------------------------------------------------------------
class MixSamp(torch.utils.data.Sampler):
    """Controls batch composition via per-source shares."""
    def __init__(self, sizes, names, shares, bs, nst, seed=0):
        self.sizes, self.names, self.bs, self.nst = sizes, names, bs, nst
        self.rng = np.random.default_rng(seed)
        per_src = [max(1, int(round(bs * shares.get(n, 0.0)))) for n in names]
        total = sum(per_src)
        if total != bs:
            per_src[int(np.argmax(per_src))] += (bs - total)
        self.per_src = per_src
        self.offsets = [0]
        for s in sizes[:-1]:
            self.offsets.append(self.offsets[-1] + s)
    def __len__(self): return self.nst
    def __iter__(self):
        for _ in range(self.nst):
            batch = []
            for off, size, n in zip(self.offsets, self.sizes, self.per_src):
                if n <= 0 or size <= 0: continue
                idxs = self.rng.integers(0, size, size=n)
                batch.extend([off + int(i) for i in idxs])
            self.rng.shuffle(batch)
            yield batch

def collate_m(batch):
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]),
            torch.stack([b[3] for b in batch]),
            [b[4] for b in batch],
            torch.stack([b[5] for b in batch])
            )

def mk_sw(sr):
    """Per-sample source weight tensor."""
    return torch.tensor([SOURCE_WEIGHTS.get(s, 0.0) for s in sr], dtype=torch.float32)

print("OK Data pipeline ready")

SC MixUp pool: 739 labeled windows
Focal secondary labels: 4372 files
OK Data pipeline ready


# S5 — Training Loop

* Training recipe¶  
Each epoch:

    1. Sample batches with 90% focal / 10% soundscape composition
    2. Compute mel spectrogram on GPU + apply SpecAugment
    3. Forward pass: backbone produces features for both the distillation head and SED head (on detached features)
    4. Loss = BCE_classification + α × MSE_distillation
    5. Validate on held-out soundscape windows using clip+frame blend
* Perch distillation mechanics  
For each batch, the frozen Perch ONNX model processes the same 5-second waveforms to produce target embeddings. The student’s distillation head projects GAP’d backbone features to 1536-d and is trained to match Perch via MSE. Since Perch trained on 14,795 species with millions of hours, this distillation teaches the small EfficientNet-B0 backbone to produce rich, generalizable audio representations.

* Learning rate schedule  
Linear warmup over 2 epochs, then cosine decay to 1e-6. Gradient clipping at norm 1.0.

In [10]:
model_teacher_path = "/content/drive/MyDrive/kaggle/Birdclef2026/outputs/exp03/"
MODELS = [model_teacher_path + f'souped_fold{i}_best_macro_auc.pt' for i in range(5)]
models_teacher = []
for path in MODELS:
    model_teacher = make_model()
    model_teacher.load_state_dict(torch.load(path), strict=False)

    models_teacher.append(model_teacher.to(device))

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

V2 backbone: (1, 1280, 8, 10)  (C=1280)
V2 backbone: (1, 1280, 8, 10)  (C=1280)
V2 backbone: (1, 1280, 8, 10)  (C=1280)
V2 backbone: (1, 1280, 8, 10)  (C=1280)
V2 backbone: (1, 1280, 8, 10)  (C=1280)


In [11]:
# =================================================================
# S5 -- TRAINING --- 5位のpseudo labeling (secondary labelの付与) ---
# =================================================================

def _load_val_waveforms(val_sc_df):
    """Load validation waveforms (always 5s)."""
    sc_file_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "soundscape_file_meta.csv")
    sc_file_dict = dict(zip(sc_file_meta["filename"], sc_file_meta["cache_file"]))
    wavs = []
    for _, row in val_sc_df.iterrows():
        cf = sc_file_dict.get(row["filename"])
        if cf is not None:
            w = load_sc_waveform_from(WAVEFORM_CACHE_DIR, cf)
            if w is not None:
                chunk = extract_chunk_np(w, int(row["start_sec"]) * SR, VAL_SAMPLES)
                wavs.append(torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0))
            else: wavs.append(torch.zeros(1, VAL_SAMPLES))
        else: wavs.append(torch.zeros(1, VAL_SAMPLES))
    return wavs

def _predict_from_waveforms(model, mel_transform, wav_list, batch_size=64):
    """Inference: mel -> model -> sigmoid. Distillation head is NOT used."""
    model.eval()
    preds_clip, preds_fmax, preds_blend = [], [], []
    with torch.no_grad():
        for s in range(0, len(wav_list), batch_size):
            batch = torch.stack(wav_list[s:s+batch_size]).to(device)
            mel = mel_transform(batch)
            B = mel.size(0)
            for i in range(B):
                mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
            with autocast():
                clip_logits, framewise = model(mel, return_framewise=True)
                frame_max = framewise.max(dim=1).values
                p_clip = torch.sigmoid(clip_logits).cpu().numpy()
                p_fmax = torch.sigmoid(frame_max).cpu().numpy()
                p_blend = 0.5 * p_clip + 0.5 * p_fmax
            preds_clip.append(p_clip); preds_fmax.append(p_fmax); preds_blend.append(p_blend)
    return {"clip": np.concatenate(preds_clip), "fmax": np.concatenate(preds_fmax),
            "blend": np.concatenate(preds_blend)}

def build_active_datasets(fold_k):
    items = []
    if USE_FOCAL:
        fds = FocalDS(audio_cache_meta[audio_cache_meta["fold"] != fold_k],
                      LABEL2IDX, secondary_lookup=focal_secondary_labels,
                      sc_mixup_sources=sc_mixup_sources if USE_FOCAL_SC_MIXUP else None,
                      fold_k=fold_k, aug=True)
        items.append(("focal", fds, len(fds)))
    if USE_LABELED_SC:
        vm = sc_cache_meta["fold"].values == fold_k
        sc_train_df = sc_cache_meta[~vm].reset_index(drop=True)
        Y_tr = Y_SC[~vm]
        sds = ScDS(Y_tr, sc_train_df, aug=True)
        items.append(("sc", sds, len(sds)))
    return items

def train_fold(fold_k):
    vm = sc_cache_meta["fold"].values == fold_k
    Y_val = Y_SC[vm]
    ns22_val = non_s22_mask_sc[vm]
    val_sc_df = sc_cache_meta[vm].reset_index(drop=True)

    active = build_active_datasets(fold_k)
    names, datasets, sizes = zip(*active)
    mds = ConcatDataset(list(datasets))
    nst = max(100, int(sum(sizes) / BATCH))

    print(f"  Streams: {dict(zip(names, sizes))}  steps/ep: {nst}")

    m = make_model()
    mel_transform = MelSpecTransform().to(device)
    spec_augment = SpecAugment().to(device)
    perch_teacher = PerchTeacher(PERCH_ONNX_PATH,
                                  "cuda" if torch.cuda.is_available() else "cpu") \
                    if USE_PERCH_DISTILL else None

    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    scaler = GradScaler()
    warmup_steps = nst * WARMUP_EPOCHS
    total_steps  = nst * EPOCHS
    warmup_sched = torch.optim.lr_scheduler.LinearLR(opt, start_factor=1/25, end_factor=1.0,
                                                      total_iters=warmup_steps)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps - warmup_steps,
                                                               eta_min=1e-6)
    sch = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_steps])

    history = {"ep": [], "train_loss": [], "cls_loss": [], "dist_loss": [],
               "macro": [], "ns22_macro": [],
               "ns22_Aves": [], "ns22_Amphibia": [], "ns22_Insecta": [], "ns22_Mammalia": [],
               "val_preds": []}
    best_ns22, best_state_ns22 = -1.0, None
    best_macro, best_state_macro = -1.0, None
    val_wavs = _load_val_waveforms(val_sc_df)

    # Early stopping parameters
    patience = 5
    early_stop_counter = 0

    # 全エポックのスコアと重みを保存するための空リストを用意
    all_states = []

    for ep in range(EPOCHS):
        m.train()
        smp = MixSamp(list(sizes), list(names), SHARES, BATCH, nst, seed=42 + ep)
        tl = DataLoader(mds, batch_sampler=smp, collate_fn=collate_m,
                        num_workers=0, pin_memory=True)
        el, el_cls, el_dist, nb_count = 0.0, 0.0, 0.0, 0
        t0 = time.time()

        for wav, lb, wt, mk, sr, is_soundscape in tl:
            wav, lb, wt, mk, is_soundscape = wav.to(device), lb.to(device), wt.to(device), mk.to(device), is_soundscape.to(device).unsqueeze(1)

            sw = mk_sw(sr).to(device)
            base_labels = lb
            final_labels = None

            with torch.no_grad():
                # wav = wave_augmenter(wav)
                mel = mel_transform(wav)
                B = mel.size(0)
                for i in range(B):
                    mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
                mel = spec_augment(mel)

            if USE_PSEUDO_LABEL:
                # ---------------------------------------------------
                # 1. 教師モデルによるオンザフライ推論
                # ---------------------------------------------------
                pseudo_logits_sum = None
                for model_teacher in models_teacher:
                    with torch.no_grad():
                        model_teacher.eval()
                        current_teacher_logits = model_teacher(mel)
                        if pseudo_logits_sum is None:
                            pseudo_logits_sum = current_teacher_logits
                        else:
                            pseudo_logits_sum += current_teacher_logits

                pseudo_logits = pseudo_logits_sum / len(models_teacher)
                pseudo_logits = pseudo_logits.detach().to(device)

                # ---------------------------------------------------
                # 5th place pseudo labeling logic
                # ---------------------------------------------------
                alpha = 0.5

                threshold = 0.1  # タスクやモデルの傾向に合わせて調整

                pseudo_labels = torch.sigmoid(pseudo_logits)

                # 確率が閾値未満のものは0にし、それ以外は元の確率を保持する
                pseudo_labels = torch.where(
                    pseudo_labels < threshold,
                    torch.zeros_like(pseudo_labels),
                    pseudo_labels
                )

                # パターンA: train_audio 用のブレンドラベル
                audio_labels = alpha * pseudo_labels + (1.0 - alpha) * base_labels

                # # パターンB: train_soundscapes 用のラベル
                soundscape_labels = base_labels # 教師の予測は混ぜず、用意したラベルをそのまま使う

                # フラグに従って、サンプルごとに A と B を選択する
                # is_soundscape が 1.0 なら soundscape_labels を、0.0 なら audio_labels を採用
                final_labels = is_soundscape * soundscape_labels + (1.0 - is_soundscape) * audio_labels

            with autocast():
                if USE_PERCH_DISTILL:
                    clip_logits, framewise, distill_emb = m(mel, return_framewise=True, return_distill=True)
                else:
                    clip_logits, framewise = m(mel, return_framewise=True)

                frame_max_logits = framewise.max(dim=1).values

                # ---------------------------------------------------
                # 2. ロスの計算 (using final_labels as targets)
                # ---------------------------------------------------

                # BCE loss using the blended/pseudo labels (final_labels) as targets
                bce_clip_final = F.binary_cross_entropy_with_logits(clip_logits, final_labels, reduction="none")
                bce_frame_final = F.binary_cross_entropy_with_logits(frame_max_logits, final_labels, reduction="none")

                # Combine clip and frame BCE losses per sample and per class
                loss_sed_per_sample_per_class = (0.5 * bce_clip_final + 0.5 * bce_frame_final)

                # ---------------------------------------------------
                # 3. 最終的な SED Loss の合算
                # ---------------------------------------------------
                # Apply sample weights and mask to the loss
                ps = (loss_sed_per_sample_per_class * wt * mk).sum(1) / (mk.sum(1) + 1e-8)
                cls_loss = (ps * sw).mean()

                # Distillation loss
                if USE_PERCH_DISTILL and perch_teacher is not None:
                    with torch.no_grad():
                        wav_5s = wav.squeeze(1)
                        N = wav_5s.shape[1]
                        if N > 160000:
                            start = (N - 160000) // 2
                            wav_5s = wav_5s[:, start:start + 160000]
                        elif N < 160000:
                            wav_5s = F.pad(wav_5s, (0, 160000 - N))
                        perch_emb = perch_teacher.embed(wav_5s).to(device)
                    # distill_loss = F.mse_loss(distill_emb, perch_emb)
                    distill_loss = 1.0 - F.cosine_similarity(distill_emb, perch_emb, dim=-1).mean()
                    loss = cls_loss + ALPHA_DISTILL * distill_loss
                else:
                    distill_loss = torch.tensor(0.0)
                    loss = cls_loss

            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()
            el += loss.item(); el_cls += cls_loss.item()
            el_dist += distill_loss.item(); nb_count += 1

        # Validation
        val_preds_dict = _predict_from_waveforms(m, mel_transform, val_wavs)
        val_preds = val_preds_dict["blend"]
        r = full_eval(Y_val, val_preds, ns22_val, TAXON_MASKS)
        for mode in ["clip", "fmax", "blend"]:
            r_mode = full_eval(Y_val, val_preds_dict[mode], ns22_val, TAXON_MASKS)
            r[f"ns22_{mode}"] = r_mode["non_s22_macro"]

        history["ep"].append(ep)
        history["train_loss"].append(round(el / nb_count, 5))
        history["cls_loss"].append(round(el_cls / nb_count, 5))
        history["dist_loss"].append(round(el_dist / nb_count, 5))
        history["macro"].append(r["macro_auc_all"])
        history["ns22_macro"].append(r["non_s22_macro"])
        for t in ["Aves", "Amphibia", "Insecta", "Mammalia"]:
            history[f"ns22_{t}"].append(r[f"non_s22_{t}"])
        history["val_preds"].append(val_preds.astype(np.float32))

        tag_ns22 = ""; tag_macro = ""

        current_macro = r["macro_auc_all"]
        # --- 2. モデル状態の取得と保存 ---
        # ☢☢重要: 毎エポックの重みをそのまま保存するとGPUメモリ(VRAM)が溢れるため、
        # 必ず .cpu().clone() を使ってCPUのシステムメモリ上に退避させます。
        current_state = {k: v.cpu().clone() for k, v in m.state_dict().items()}

        # リストに (スコア, 重み) のタプルを追加
        all_states.append((current_macro, current_state))

        if r["non_s22_macro"] > best_ns22:
            best_ns22 = r["non_s22_macro"]
            best_state_ns22 = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_ns22 = " *ns22"

        if r["macro_auc_all"] > best_macro:
            best_macro = r["macro_auc_all"]
            best_state_macro = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_macro = " *macro"
            early_stop_counter = 0 # Reset early stopping counter
        else:
            early_stop_counter += 1 # Increment early stopping counter

        dist_str = f" dist={el_dist/nb_count:.4f}" if USE_PERCH_DISTILL else ""
        print(f"    Ep{ep:02d}: loss={el/nb_count:.4f} cls={el_cls/nb_count:.4f}{dist_str} "
              f"lr={opt.param_groups[0]['lr']:.1e} | "
              f"ns22: {r['ns22_blend']:.4f} | "
              f"macro: {r['macro_auc_all']:.4f} | "
              f"Av={r['non_s22_Aves']:.4f} Am={r['non_s22_Amphibia']:.4f} "
              f"In={r['non_s22_Insecta']:.4f} Ma={r['non_s22_Mammalia']:.4f} "
              f"[{time.time()-t0:.0f}s]{tag_ns22}{tag_macro}")

        # Check for early stopping
        if early_stop_counter >= patience:
            print(f"    Early stopping triggered after {patience} epochs without improvement.")
            break

    print(f"--- Starting Greedy Soup based on macro_auc_all ---")

    # 1. 保存した全エポックの状態をスコア順にソート（all_statesに各エポックの情報を保存しておいたと仮定）
    all_states.sort(key=lambda x: x[0], reverse=True)

    # 実行時間節約のため上位K個に絞る
    TOP_K = 5
    all_states = all_states[:TOP_K]

    # 2. 1位のモデルで初期化
    best_soup_score, best_soup_state = all_states[0]
    current_soup_state = {k: v.clone() for k, v in best_soup_state.items()}
    num_models_in_soup = 1

    # --- Initialize a temporary model for souping evaluation ---
    temp_model_for_soup = make_model()

    # 3. 2位以降のモデルを順番にテスト
    for i in range(1, len(all_states)):
        candidate_score, candidate_state = all_states[i]

        # --- 一時的なSoupの作成 (Moving Average) ---
        temp_soup_state = {}
        for key, value in current_soup_state.items():
            if value.dtype.is_floating_point:
                # (今の総和 + 新しい候補) / (今の数 + 1)
                temp_soup_state[key] = (value * num_models_in_soup + candidate_state[key]) / (num_models_in_soup + 1)
            else:
                # BatchNormの追跡変数(int)などはベースモデルのものを維持
                temp_soup_state[key] = value.clone()

        # --- モデルに一時的な重みをロード ---
        temp_model_for_soup.load_state_dict(temp_soup_state)

        # --- ‼‼‼ここでValidationを再実行‼‼‼ ---
        # ノートブック内のバリデーション処理（予測とAUC計算）を再利用し、スコアを取得します。
        soup_val_preds_dict = _predict_from_waveforms(temp_model_for_soup, mel_transform, val_wavs)
        soup_val_preds = soup_val_preds_dict["blend"]
        soup_eval_results = full_eval(Y_val, soup_val_preds, ns22_val, TAXON_MASKS)
        temp_macro = soup_eval_results['macro_auc_all']

        # --- 採用・不採用の判定 ---
        if temp_macro >= best_soup_score:
            print(f"  Model {i} (Epoch score: {candidate_score:.4f}) added! New macro AUC: {temp_macro:.4f}")
            best_soup_score = temp_macro
            current_soup_state = temp_soup_state
            num_models_in_soup += 1
        else:
            print(f"  Model {i} rejected. Temp macro AUC: {temp_macro:.4f}")

    del perch_teacher, m, mel_transform, spec_augment, temp_model_for_soup
    torch.cuda.empty_cache(); gc.collect()
    return best_state_ns22, best_state_macro, history, current_soup_state

In [12]:
import onnxruntime as ort

# =================================================================
# S6 -- FOLD LOOP + ONNX EXPORT
# =================================================================

if MODE != "train":
    print("Skipping training (MODE='infer')")
    oof_ns22 = None
    all_hist = {}
else:

    oof_ns22 = np.full((len(sc_cache_meta), NUM_CLASSES), np.nan, dtype=np.float32)
    all_hist = {}
    for fold_k in FOLDS:
        print(f"\n{'='*60}")
        print(f"FOLD {fold_k}")
        print(f"{'='*60}")
        vm = sc_cache_meta["fold"].values == fold_k
        val_sc_df_k = sc_cache_meta[vm].reset_index(drop=True)

        best_ns22_state, best_macro_state, hist, current_soup_state = train_fold(fold_k)
        all_hist[fold_k] = hist

        mel_tf = MelSpecTransform().to(device)
        val_wavs_k = _load_val_waveforms(val_sc_df_k)

        if best_macro_state is not None:
            # Save PyTorch checkpoint
            torch.save(best_macro_state, OUT_DIR / f"{EXP}"/ f"fold{fold_k}_best_macro_auc.pt")
            torch.save(current_soup_state, OUT_DIR / f"{EXP}"/ f"souped_fold{fold_k}_best_macro_auc.pt")
            m = make_model()
            m.load_state_dict(best_macro_state, strict=False)
            oof_ns22[vm] = _predict_from_waveforms(m, mel_tf, val_wavs_k)["blend"]

            # --- ONNX Export (Conv1d remap for stable tracing) ---
            m.eval()
            INF_N_MELS = 256
            INF_N_FRAMES = VAL_SAMPLES // HOP_LENGTH + 1
            # INF_N_FRAMES = TRAIN_SAMPLES // HOP_LENGTH + 1

            class SEDExportWrapper(nn.Module):
                def __init__(self, backbone_name, num_classes, backbone_dim, hidden_dim=512):
                    super().__init__()
                    self.backbone = timm.create_model(
                        backbone_name, pretrained=False, in_chans=1,
                        num_classes=0, global_pool="", drop_path_rate=0.1,
                    )
                    self.gem_freq = GeMFreqPool(p_init=3.0)
                    self.dense_drop1 = nn.Dropout(0.25)
                    self.dense_conv = nn.Conv1d(backbone_dim, hidden_dim, 1)
                    self.dense_relu = nn.ReLU(inplace=True)
                    self.dense_drop2 = nn.Dropout(0.5)
                    self.att = nn.Conv1d(hidden_dim, num_classes, 1)
                    self.cla = nn.Conv1d(hidden_dim, num_classes, 1)

                def forward(self, mel):
                    h = self.backbone(mel)
                    h = self.gem_freq(h)
                    h = self.dense_drop1(h)
                    h = self.dense_conv(h)
                    h = self.dense_relu(h)
                    h = self.dense_drop2(h)
                    norm_att = torch.softmax(torch.tanh(self.att(h)), dim=-1)
                    framewise = self.cla(h)
                    clip = torch.sum(norm_att * framewise, dim=2)
                    return clip, framewise.permute(0, 2, 1)

            def load_and_remap_state(export_model, trained_state):
                remap = {}
                for k, v in trained_state.items():
                    if k.startswith("distill_head."):
                        continue
                    if k == "dense.1.weight":
                        remap["dense_conv.weight"] = v.unsqueeze(-1)
                    elif k == "dense.1.bias":
                        remap["dense_conv.bias"] = v
                    else:
                        remap[k] = v
                export_model.load_state_dict(remap, strict=False)

            export_model = SEDExportWrapper(
                BACKBONE_NAME, NUM_CLASSES, m.backbone_dim
            ).to(device)
            # Original model
            load_and_remap_state(export_model, best_macro_state)
            export_model.eval()

            dummy_mel = torch.randn(1, 1, INF_N_MELS, INF_N_FRAMES).to(device)
            onnx_path = OUT_DIR / f"{EXP}"/ f"stage1_sed_distill_fold{fold_k}.onnx"
            torch.onnx.export(
                export_model, dummy_mel, str(onnx_path),
                input_names=["mel"],
                output_names=["clip_logits", "framewise_logits"],
                dynamic_axes={"mel": {0: "batch"},
                              "clip_logits": {0: "batch"},
                              "framewise_logits": {0: "batch"}},
                opset_version=18,
            )

            # Souped model
            load_and_remap_state(export_model, current_soup_state)
            export_model.eval()

            dummy_mel = torch.randn(1, 1, INF_N_MELS, INF_N_FRAMES).to(device)
            onnx_path = OUT_DIR / f"{EXP}"/ f"stage1_souped_sed_distill_fold{fold_k}.onnx"
            torch.onnx.export(
                export_model, dummy_mel, str(onnx_path),
                input_names=["mel"],
                output_names=["clip_logits", "framewise_logits"],
                dynamic_axes={"mel": {0: "batch"},
                              "clip_logits": {0: "batch"},
                              "framewise_logits": {0: "batch"}},
                opset_version=18,
            )

            # Verify
            _sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
            _onnx_out = _sess.run(None, {'mel': dummy_mel.cpu().numpy()})
            with torch.no_grad():
                _ref_clip, _ref_frame = export_model(dummy_mel)
            _diff = np.abs(_ref_clip.cpu().numpy() - _onnx_out[0]).max()
            print(f"  ONNX verify: max|diff|={_diff:.3e}")
            # assert _diff < 2e-3, f"ONNX export diverged: {_diff}" # Increased tolerance
            del _sess

            size_mb = onnx_path.stat().st_size / 1e6
            print(f"  Exported {onnx_path.name} ({size_mb:.1f} MB)")
            del m, export_model



FOLD 0
  Streams: {'focal': 28906, 'sc': 584}  steps/ep: 460
V2 backbone: (1, 1280, 8, 10)  (C=1280)
Perch ONNX loaded: embed_idx=0
    Ep00: loss=0.9083 cls=0.2615 dist=0.6468 lr=2.6e-04 | ns22: 0.5099 | macro: 0.5724 | Av=0.4843 Am=0.7388 In=0.4194 Ma=0.2000 [379s] *ns22 *macro
    Ep01: loss=0.4528 cls=0.0424 dist=0.4104 lr=5.0e-04 | ns22: 0.7977 | macro: 0.8538 | Av=0.9388 Am=0.7393 In=0.7673 Ma=1.0000 [192s] *ns22 *macro
    Ep02: loss=0.3373 cls=0.0333 dist=0.3041 lr=5.0e-04 | ns22: 0.8359 | macro: 0.8831 | Av=0.9559 Am=0.7713 In=0.8144 Ma=1.0000 [151s] *ns22 *macro
    Ep03: loss=0.2868 cls=0.0300 dist=0.2568 lr=4.9e-04 | ns22: 0.8628 | macro: 0.9017 | Av=0.9647 Am=0.8484 In=0.8248 Ma=1.0000 [140s] *ns22 *macro
    Ep04: loss=0.2586 cls=0.0283 dist=0.2303 lr=4.8e-04 | ns22: 0.8747 | macro: 0.9036 | Av=0.9740 Am=0.8546 In=0.8422 Ma=1.0000 [136s] *ns22 *macro
    Ep05: loss=0.2407 cls=0.0272 dist=0.2135 lr=4.6e-04 | ns22: 0.8821 | macro: 0.9059 | Av=0.9617 Am=0.8864 In=0.8383 Ma=

W0505 03:42:48.404000 823 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0505 03:42:48.404000 823 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0505 03:42:48.405000 823 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0505 03:42:48.406000 823 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


W0505 03:42:52.811000 823 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0505 03:42:52.812000 823 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0505 03:42:52.813000 823 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0505 03:42:52.813000 823 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
  ONNX verify: max|diff|=1.694e-03
  Exported stage1_souped_sed_distill_fold0.onnx (0.6 MB)


* ns22: サイト「S22」を除外した検証用Soundscapeデータにおける、予測ブレンド（Clip予測とFrame-max予測の平均）のMacro AUCスコアです。S22はラベルノイズが多い既知のサイトであるため、これを除外したこのスコアが主力の評価指標として機能しています。  

* 分類（Taxonomy）ごとのAUCスコア: S22を除外したデータにおける、生物分類ごとのMacro AUCスコアです。  

    * Av: 鳥類（Aves）

    * Am: 両生類（Amphibia）

    * In: 昆虫類（Insecta）

    * Ma: 哺乳類（Mammalia）

# S7 — OOF Evaluation

In [13]:
# =================================================================
# S7 -- EVALUATION SUMMARY
# =================================================================

if MODE != "train":
    print("Skipping evaluation (MODE='infer')")
else:

    has = ~np.isnan(oof_ns22[:, 0])
    if has.sum() > 0:
        r_all = full_eval(Y_SC[has], oof_ns22[has], non_s22_mask_sc[has], TAXON_MASKS)
        print("=" * 60)
        print("OOF RESULTS (best-ns22 checkpoints)")
        print("=" * 60)
        print(f"  macro AUC (all):        {r_all['macro_auc_all']:.4f}")
        print(f"  macro AUC (non-S22):    {r_all['non_s22_macro']:.4f}")
        for t in ["Aves", "Amphibia", "Insecta", "Mammalia"]:
            print(f"    {t:<12}: {r_all.get(f'non_s22_{t}', float('nan')):.4f}")

    # Per-epoch progression
    print("\nPer-epoch pooled non-S22 AUC:")
    fold_true, fold_ns22_m = {}, {}
    for fk in range(N_FOLDS):
        vm = sc_cache_meta["fold"].values == fk
        fold_true[fk] = Y_SC[vm]
        fold_ns22_m[fk] = non_s22_mask_sc[vm]

    n_eps = [len(all_hist[k]["val_preds"]) for k in range(N_FOLDS) if k in all_hist]
    max_ep = min(n_eps) if n_eps else 0
    for ep in range(max_ep):
        pp = np.concatenate([all_hist[k]["val_preds"][ep] for k in range(N_FOLDS) if k in all_hist])
        pt = np.concatenate([fold_true[k] for k in range(N_FOLDS) if k in all_hist])
        pm = np.concatenate([fold_ns22_m[k] for k in range(N_FOLDS) if k in all_hist])
        ns, _ = compute_macro_auc(pt, pp, mask=pm)
        print(f"  Ep{ep:02d}: {ns:.4f}")

OOF RESULTS (best-ns22 checkpoints)
  macro AUC (all):        0.9239
  macro AUC (non-S22):    0.9057
    Aves        : 0.9763
    Amphibia    : 0.8735
    Insecta     : 0.9008
    Mammalia    : 1.0000

Per-epoch pooled non-S22 AUC:
  Ep00: 0.5099
  Ep01: 0.7977
  Ep02: 0.8359
  Ep03: 0.8628
  Ep04: 0.8747
  Ep05: 0.8821
  Ep06: 0.8645
  Ep07: 0.8926
  Ep08: 0.9025
  Ep09: 0.8957
  Ep10: 0.9029
  Ep11: 0.9057
  Ep12: 0.8931
  Ep13: 0.8920
  Ep14: 0.8879
  Ep15: 0.8921
  Ep16: 0.8926


In [14]:
import time
from google.colab import runtime

# 1時間（X秒）待機してから終了する場合
time.sleep(10)
runtime.unassign()